# 09. Benchmark

> Pull together cross-validated results from the classical (`06_classification`), deep (`07_deep_learning`), and regression (`08_finger_regression`) pipelines, and place them next to published numbers on the same paradigm.

References include high-gamma + LDA pipelines, Riemannian methods, and EEGNet/ShallowConvNet papers. Highlights where this pipeline matches state-of-the-art and where it lags. No code is exported — this notebook is a results dashboard.

## Setup

In [ ]:
%config InlineBackend.figure_format = 'retina'

import numpy as np
import matplotlib.pyplot as plt
from br41n_ecog_hand_pose.data import load_ecog, FINGER_NAMES
from br41n_ecog_hand_pose.preprocessing import preprocess
from br41n_ecog_hand_pose.epoching import epoch_recording
from br41n_ecog_hand_pose.features import multi_band_power
from br41n_ecog_hand_pose.classification import CLASSIFIERS, cross_validate
from br41n_ecog_hand_pose.regression import glove_targets, cross_validate_regression

plt.rcParams.update({
    'axes.grid':      True,
    'grid.linestyle': ':',
    'grid.linewidth': 0.5,
    'grid.alpha':     0.6,
})

## Build the feature matrix once

Every pipeline below reuses the same preprocessed epochs and band-power features so differences are attributable to the model, not the input.

In [ ]:
#| eval: false
rec = load_ecog()
clean, bad = preprocess(rec.ecog, rec.fs)
epochs, classes = epoch_recording(rec, tmin=0.0, tmax=2.0, signal=clean)
X = multi_band_power(epochs, rec.fs)
Y, _ = glove_targets(rec, tmin=0.0, tmax=2.0)
print(f'epochs {epochs.shape}, X {X.shape}, Y {Y.shape}, bad channels: {bad.tolist()}')

## Classification — 3-class gesture decoding

In [ ]:
#| eval: false
clf_results = {}
for name, make_clf in CLASSIFIERS.items():
    accs, cm, _ = cross_validate(make_clf, X, classes)
    clf_results[name] = (accs.mean(), accs.std())
    print(f'{name:>8}: {accs.mean():.3f} \u00b1 {accs.std():.3f}')

## Regression — continuous finger flexion

In [ ]:
#| eval: false
corrs, _ = cross_validate_regression(X, Y, alpha=1.0)
for fi, name in enumerate(FINGER_NAMES):
    print(f'  {name:>7}: r = {corrs[:, fi].mean():.3f} \u00b1 {corrs[:, fi].std():.3f}')
print(f'  {"average":>7}: r = {corrs.mean():.3f}')

## Summary plot

In [ ]:
#| eval: false
fig, axs = plt.subplots(1, 2, figsize=(11, 3.5))

names = list(clf_results)
means = [clf_results[n][0] for n in names]
stds  = [clf_results[n][1] for n in names]
axs[0].bar(names, means, yerr=stds, capsize=3)
axs[0].axhline(1/3, color='k', lw=0.8, ls='--', label='chance (3-class)')
axs[0].set_ylim(0, 1); axs[0].set_ylabel('accuracy')
axs[0].set_title('Gesture classification (5-fold CV)')
axs[0].legend(fontsize=8)

axs[1].bar(FINGER_NAMES, corrs.mean(axis=0), yerr=corrs.std(axis=0), capsize=3)
axs[1].axhline(0, color='k', lw=0.8)
axs[1].set_ylim(-0.1, 1)
axs[1].set_ylabel('Pearson r')
axs[1].set_title('Finger-flexion regression (5-fold CV)')
plt.tight_layout(); plt.show()

## Comparison with published baselines

Direct comparisons are imperfect — different subjects, paradigms, channel counts, and trial counts mean the numbers below are *typical ranges*, not strict targets.

### Gesture classification

| Pipeline | Reported accuracy | Notes |
|---|---|---|
| **Our LDA + log-band-power** | see run above | 60-channel ECoG, 3-class, 90 trials, stratified 5-fold |
| Hand-gesture ECoG with HG-band features + LDA (Pistohl et al. 2008, Wang et al. 2013) | 0.80–0.95 | Subject- and session-dependent |
| EEGNet on motor imagery (Lawhern et al. 2018) | 0.65–0.75 (4-class EEG) | Lower because EEG SNR is worse than ECoG |
| ShallowConvNet, motor-decoding ECoG (Schirrmeister et al. 2017) | 0.80–0.92 | Within ~3% of CSP+LDA on ECoG |
| Riemannian (covariance + tangent space) (Yger et al. 2017) | 0.80–0.90 | Strong without explicit feature engineering |

### Continuous finger decoding

| Pipeline | Reported per-finger r | Notes |
|---|---|---|
| **Our Ridge + log-band-power** | see run above | Mean of 5 fingers, 5-fold CV |
| Linear filter from HG band power, BCI Competition IV-4 (Liang & Bougrain 2012) | 0.45–0.60 | The standard reference for ECoG finger decoding |
| Deep CNN on raw ECoG (Xie et al. 2018) | 0.55–0.70 | ~5–10 pp gain over linear baselines |

### Where this pipeline is expected to lag

- **EEGNet** on 90 trials almost always underperforms LDA + HG-band features on ECoG — there's not enough data to fit a CNN well. With more sessions or data augmentation it would close the gap.
- **Riemannian** isn't implemented here; it would be a quick add via `pyriemann` and is worth comparing on the same folds.
- **CSP** features per band would likely give a small bump (1–3 pp) over raw band power for the gesture task; not implemented in `05_features` yet.